# Model-only ranking

Rank a set of simple time-series models on the same indicator and compute rank confidence intervals among them. Models are evaluated in **real time**: at survey quarter `S` we use only the RTDSM vintage published in `S`, restricted to data through `S-1`.

Pipeline:
1. Load SPF microdata (only to obtain a consistent set of target quarters from its top-N) and the RTDSM for the indicator.
2. Define a set of forecasting models.
3. Generate model forecasts at every target quarter, with the same vintage discipline.
4. Run stepwise + marginal rank CIs on the model-only panel.

In [1]:
import numpy as np
import pandas as pd
from functools import partial

from rankci import (
    select_top_forecasters,
    compute_pairwise,
    rank_ci_marginal_pairwise,
    rank_ci_stepwise_pairwise,
    rank_confidence_intervals_simulation,
    rank_confidence_intervals_simulation_pairwise,
    rank_ci_stepwise_simulation,
    rank_ci_stepwise_simulation_pairwise,
    rank_ci_marginal_simulation_pairwise,
)
from rankci.data.philly import load_spf, load_rtdsm, compute_error_panel
from rankci.models.philly import (
    model_error_panel,
    forecast_naive,
    forecast_ar1,
    forecast_ar,
    forecast_rw_drift,
    forecast_ma4,
    forecast_historical_mean,
)

# Configuration

Set the indicator and parameters here. Switch indicator/RTDSM and the rest of the notebook follows.

In [2]:
# --- Indicator ---
INDICATOR     = "NGDP"
RTDSM_PATH    = "../../data/philly/NOUTPUTQvQd.xlsx"
RTDSM_PREFIX  = "NOUTPUT"
RTDSM_FREQ    = "quarterly"

# Alternative: unemployment
# INDICATOR    = "UNEMP"
# RTDSM_PATH   = "../../data/philly/rucQvMd.xlsx"
# RTDSM_PREFIX = "RUC"
# RTDSM_FREQ   = "monthly"

# --- Comparison knobs ---
HORIZON   = 3              # SPF horizon (3 = one-quarter-ahead)
METRIC    = "squared"      # "squared" → MSE, "absolute" → MAE
N_SPF     = 8              # how many SPF forecasters to keep
ALPHA     = 0.2           # 1-alpha confidence
B         = 5000           # bootstrap replications
SEED      = 42

# Load data and pick the target-quarter grid

We use the SPF top-N panel only to obtain the same evaluation window as the SPF-based notebooks — the SPF columns themselves are NOT included in the ranking.

In [3]:
df    = load_spf("../../data/philly/SPFmicrodata.xlsx", sheet=INDICATOR)
rtdsm = load_rtdsm(RTDSM_PATH, prefix=RTDSM_PREFIX, freq=RTDSM_FREQ)

X_spf_full = compute_error_panel(df, rtdsm, indicator=INDICATOR, horizon=HORIZON, metric=METRIC)
X_spf_top  = select_top_forecasters(X_spf_full, N=N_SPF, min_obs=20)
target_quarters = list(X_spf_top.index)

print(f"SPF panel: {X_spf_full.shape[0]} target quarters × {X_spf_full.shape[1]} forecasters total")
print(f"Using top-{N_SPF} quarter coverage as the evaluation grid: "
      f"{len(target_quarters)} target quarters "
      f"({target_quarters[0]} → {target_quarters[-1]})")

/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


SPF panel: 225 target quarters × 345 forecasters total
Using top-8 quarter coverage as the evaluation grid: 225 target quarters ((1968, 4) → (2025, 1))


# Define and run the models

Add or remove models here. Each entry is `name → callable(history, h)`. Use `partial` to bind extra kwargs.

In [4]:
models = {
    "naive":            forecast_naive,
    "rw_drift":         forecast_rw_drift,
    "ma4":              forecast_ma4,
    "historical_mean":  forecast_historical_mean,
    "ar1_levels":       partial(forecast_ar1, transform="levels"),
    "ar1_log_diff":     partial(forecast_ar1, transform="log_diff"),
    "ar2_log_diff":     partial(forecast_ar, lags=2, transform="log_diff"),
}

X_models = model_error_panel(
    rtdsm,
    models,
    target_quarters,
    spf_horizon=HORIZON,
    metric=METRIC,
)

# Drop quarters where any model returned NaN (typically early-sample warmup)
X_models = X_models.dropna(how="any")

print(f"Model panel: {X_models.shape[0]} quarters × {X_models.shape[1]} models")
print(f"\nModel mean {METRIC} error:")
print(X_models.mean().round(2).sort_values())

Model panel: 224 quarters × 7 models

Model mean squared error:
ar1_levels            87872.84
ar2_log_diff         120514.19
ar1_log_diff         122080.54
rw_drift             134141.77
naive                183451.15
ma4                  389640.15
historical_mean    81695360.12
dtype: float64


# Rank the models

Apply both the stepwise (simultaneous) and marginal (per-model) procedures to the model-only panel.

In [5]:
X = X_models.values
ids = list(X_models.columns)

metric_col = "MSE" if METRIC == "squared" else "MAE"

# --- Stepwise (simultaneous) ---
out_step = rank_ci_stepwise_pairwise(X, alpha=ALPHA, B=B, seed=SEED, verbose=False)

# --- Marginal (per-model) ---
out_marg = rank_ci_marginal_pairwise(X, alpha=ALPHA, B=B, seed=SEED)

results = pd.DataFrame({
    "model":      ids,
    metric_col:   out_step["theta_hat"].round(2),
    "RMSE":       np.sqrt(out_step["theta_hat"]).round(2),
    "CI_step":    [f"[{l},{u}]" for l, u in out_step["rank_ci"]],
    "CI_marg":    [f"[{l},{u}]" for l, u in out_marg["rank_ci"]],
    "cv_j_marg":  out_marg["critical_values"].round(3),
}).sort_values(metric_col).reset_index(drop=True)
results.index += 1
results.index.name = "Rank"
results

,model,MSE,RMSE,CI_step,CI_marg,cv_j_marg
Rank,,,,,,
1,ar1_levels,87872.84,296.43,"[1,3]","[1,3]",1.417
2,ar2_log_diff,120514.19,347.15,"[1,6]","[1,5]",2.111
3,ar1_log_diff,122080.54,349.40,"[1,6]","[1,5]",2.090
4,rw_drift,134141.77,366.25,"[2,4]","[2,4]",1.410
5,naive,183451.15,428.31,"[3,5]","[3,5]",1.364
6,ma4,389640.15,624.21,"[4,6]","[6,6]",1.203
7,historical_mean,81695360.12,9038.55,"[7,7]","[7,7]",0.964


# Joint ranking: SPF top-N (complete cases) + models

Now include the top-N SPF forecasters in the ranking alongside the models, but restricted to the **complete-cases** subset where every top-N SPF forecaster is observed. Aligning on the model panel as well gives a fully balanced joint panel — no NaN anywhere — so the simulation methods apply cleanly (no MDS/PSD projection) and the marginal-vs-pairwise rank ambiguity disappears.

Three CIs are reported on the same panel for each forecaster/model:

- **CI_sim_wald** — Wald-style simulation (single critical value)
- **CI_sim_step** — stepwise simulation refinement
- **CI_boot_step** — bootstrap stepwise (reference)

In [6]:
# Restrict to quarters where every top-N SPF forecaster is observed
X_spf_cc = X_spf_top.dropna(how="any")
print(f"SPF complete cases:    {X_spf_cc.shape[0]} quarters × {X_spf_cc.shape[1]} forecasters")
print(f"  ({X_spf_cc.index[0]} → {X_spf_cc.index[-1]})")

# Align with the model panel on the same quarter index
common_idx = X_spf_cc.index.intersection(X_models.index)
X_full_cc = pd.concat(
    [X_spf_cc.loc[common_idx], X_models.loc[common_idx]],
    axis=1,
).dropna(how="any")
print(f"Joint balanced panel:  {X_full_cc.shape[0]} quarters × {X_full_cc.shape[1]} (SPF + models)")
print(f"  Columns: {X_full_cc.columns.tolist()}")

SPF complete cases:    21 quarters × 8 forecasters
  ((1991, 1) → (2003, 3))
Joint balanced panel:  20 quarters × 15 (SPF + models)
  Columns: [65, 426, 84, 433, 428, 411, 421, 40, 'naive', 'rw_drift', 'ma4', 'historical_mean', 'ar1_levels', 'ar1_log_diff', 'ar2_log_diff']


In [7]:
import time

X = X_full_cc.values
ids = [str(c) for c in X_full_cc.columns]
metric_col = "MSE" if METRIC == "squared" else "MAE"

# Wald simulation (complete-cases)
t0 = time.perf_counter()
out_sim_wald = rank_confidence_intervals_simulation(X, alpha=ALPHA, B=B, seed=SEED)
t_sim_wald = time.perf_counter() - t0

# Stepwise simulation (complete-cases)
t0 = time.perf_counter()
out_sim_step = rank_ci_stepwise_simulation(X, alpha=ALPHA, B=B, seed=SEED)
t_sim_step = time.perf_counter() - t0

# Bootstrap stepwise (reference)
t0 = time.perf_counter()
out_boot = rank_ci_stepwise_pairwise(X, alpha=ALPHA, B=B, seed=SEED, verbose=False)
t_boot = time.perf_counter() - t0

# Pair rank (on a balanced panel this coincides with MSE_rank by construction)
theta = X.mean(axis=0)
delta = theta[:, None] - theta[None, :]
pair_rank = (delta > 0).sum(axis=1) + 1

results = pd.DataFrame({
    "ID":           ids,
    "kind":         ["model" if c in X_models.columns else "SPF" for c in X_full_cc.columns],
    metric_col:     theta.round(2),
    "pair_rank":    pair_rank,
    "CI_sim_wald":  [f"[{l},{u}]" for l, u in out_sim_wald["rank_ci"]],
    "CI_sim_step":  [f"[{l},{u}]" for l, u in out_sim_step["rank_ci"]],
    "CI_boot_step": [f"[{l},{u}]" for l, u in out_boot["rank_ci"]],
}).sort_values(metric_col).reset_index(drop=True)
results.index += 1
results.index.name = f"{metric_col}_rank"

print(f"alpha = {ALPHA}, B = {B}, joint complete-cases n = {X_full_cc.shape[0]}")
print(f"timings: sim_wald={t_sim_wald:.3f}s, sim_step={t_sim_step:.3f}s, boot_step={t_boot:.3f}s")
print(f"sim_wald critical value: {out_sim_wald['critical_value']:.3f}")
print()
results

alpha = 0.2, B = 5000, joint complete-cases n = 20
timings: sim_wald=0.054s, sim_step=0.629s, boot_step=20.414s
sim_wald critical value: 2.404



,ID,kind,MSE,pair_rank,CI_sim_wald,CI_sim_step,CI_boot_step
MSE_rank,,,,,,,
1,84,SPF,7812.84,1,"[1,12]","[1,12]","[1,12]"
2,40,SPF,8622.88,2,"[1,12]","[1,12]","[1,12]"
3,ar1_levels,model,9404.64,3,"[1,12]","[1,12]","[1,13]"
4,411,SPF,9451.49,4,"[1,11]","[1,11]","[1,12]"
5,ar2_log_diff,model,9594.25,5,"[1,12]","[1,12]","[1,13]"
6,433,SPF,10023.16,6,"[1,11]","[1,11]","[1,12]"
7,426,SPF,10265.33,7,"[1,11]","[1,11]","[1,11]"
8,ar1_log_diff,model,10342.10,8,"[1,12]","[1,12]","[1,13]"
9,428,SPF,10797.78,9,"[1,12]","[1,12]","[1,12]"
